In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np

from astropy.io import ascii

from astropy.coordinates import ICRS, AltAz, Angle, EarthLocation, SkyCoord, get_sun
import astropy.units as u
from astropy.time import Time
import lsst.summit.utils.butlerUtils as butlerUtils
from lsst.daf.butler import DatasetNotFoundError
import lsst_efd_client

from lsst.ts.ofc import OFC, OFCData

import logging
import smplotlib

In [ ]:
logging.basicConfig(level=logging.DEBUG)

In [ ]:
log = logging.getLogger("20260404")

In [ ]:
log.setLevel(logging.DEBUG)

In [ ]:
efd_client = lsst_efd_client.EfdClient("summit_efd")

In [ ]:
start = Time(
    "2026-04-04T23:00:00",
    format="isot",
    scale="utc",
)
end = Time(
    "2026-04-05T10:00:00",
    format="isot",
    scale="utc",
)

In [ ]:
dayobs = 2026040400000

In [ ]:
wavefront_errors = await efd_client.select_time_series(
    "lsst.sal.MTAOS.logevent_wavefrontError",
    [
        f"nollZernikeValues{i}" for i in range(27)
    ] + [
        f"nollZernikeIndices{i}" for i in range(27)
    ] + [
        "sensorId",
        "visitId"
    ],
    start=start,
    end=end,
)

In [ ]:
wavefront_errors.iloc[4]

In [ ]:
4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 20, 21, 22, 27, 28

In [ ]:
degrees_of_freedom = await efd_client.select_time_series(
    "lsst.sal.MTAOS.logevent_degreeOfFreedom",
    [
        f"aggregatedDoF{i}" for i in range(50)
    ] + [
        f"visitDoF{i}" for i in range(50)
    ] + [
        "visitId",
    ] + [
        f"kpGain{i}" for i in range(50)
    ]+ [
        f"kiGain{i}" for i in range(50)
    ],
    start=start,
    end=end,
)

In [ ]:
camera_hex_corr = await efd_client.select_time_series(
    "lsst.sal.MTAOS.logevent_cameraHexapodCorrection",
    ["x", "y", "z", "visitId"],
    start=start,
    end=end,
)

m2_hex_corr = await efd_client.select_time_series(
    "lsst.sal.MTAOS.logevent_m2HexapodCorrection",
    ["x", "y", "z", "visitId"],
    start=start,
    end=end,
)

In [ ]:
wavefront_errors["nollZernikeValues0"].iloc[0]

In [ ]:
fig = plt.figure(1, (11,4))
fig.add_subplot(111)

gain = []

plt.plot(
    wavefront_errors["visitId"]-dayobs,
    wavefront_errors["nollZernikeValues0"],
    ".",
)
ylim = (-1, 0.5)
for g in gain:
    plt.fill_between(g, [ylim[0], ylim[0]], [ylim[1],ylim[1]], alpha=0.1)
plt.grid()
plt.ylim(*ylim)
# plt.xlim(0, 100)

plt.ylabel("Z4")
plt.xlabel("Sequence Number")
# fig.add_subplot(312)

# plt.plot(
#     wavefront_errors["visitId"]-2026040300000,
#     wavefront_errors["nollZernikeValues1"],
#     ".",
# )
# plt.grid()
# plt.ylim(-1.0, 0.5)
# plt.xlim(0, 100)

# fig.add_subplot(313)

# plt.plot(
#     wavefront_errors["visitId"]-2026040300000,
#     wavefront_errors["nollZernikeValues2"],
#     ".",
# )
# plt.grid()
# plt.ylim(-1.0, 0.5)
# plt.xlim(0, 100)


In [ ]:
fig = plt.figure(1)
fig.add_subplot(311)
plt.plot(
    degrees_of_freedom["aggregatedDoF0"],
    ".",
)

fig.add_subplot(312)
plt.plot(
    degrees_of_freedom["aggregatedDoF1"],
    ".",
)

fig.add_subplot(313)
plt.plot(
    degrees_of_freedom["aggregatedDoF2"],
    ".",
)

In [ ]:
fig = plt.figure(1)
fig.add_subplot(311)
plt.plot(
    degrees_of_freedom["visitDoF0"],
    ".",
)
xlim = plt.xlim()
plt.plot(xlim, [0, 0], ":")
plt.xlim(xlim)

fig.add_subplot(312)
plt.plot(
    degrees_of_freedom["visitDoF1"],
    ".",
)

fig.add_subplot(313)
plt.plot(
    degrees_of_freedom["visitDoF2"],
    ".",
)

In [ ]:
fig = plt.figure(1)
ax1 = fig.add_subplot(311)
plt.plot(
    degrees_of_freedom["visitId"]-dayobs,
    degrees_of_freedom["visitDoF0"],
    ".",
)
plt.xlim(0,700)
plt.ylim(-100,100)

plt.ylabel("DoF 0")
plt.title("Visit DoF")
plt.grid()
ax1.tick_params(axis='x', labelbottom=False)

ax2 = fig.add_subplot(312)
plt.plot(
    degrees_of_freedom["visitId"]-dayobs,
    degrees_of_freedom["visitDoF5"],
    ".",
)
plt.xlim(0,700)
plt.ylim(-200, 200)
plt.xlabel("Time - UTC")
plt.ylabel("DoF 5")
plt.grid()

fig.subplots_adjust(hspace = 0)

In [ ]:
fig = plt.figure(1)
ax1 = fig.add_subplot(311)
plt.plot(
    degrees_of_freedom["visitId"]-dayobs,
    degrees_of_freedom["kpGain0"],
    ".",
)
plt.xlim(0,700)
#plt.ylim(-100,100)

plt.ylabel("DoF 0")
plt.title("Visit DoF")
plt.grid()
ax1.tick_params(axis='x', labelbottom=False)

ax2 = fig.add_subplot(312)
plt.plot(
    degrees_of_freedom["visitId"]-dayobs,
    degrees_of_freedom["kiGain0"],
    ".",
)
plt.xlim(0,700)
#plt.ylim(-200, 200)
plt.xlabel("Time - UTC")
plt.ylabel("kiGain")
plt.grid()

fig.subplots_adjust(hspace = 0)

In [ ]:
fig = plt.figure(1)
fig.add_subplot(311)
plt.plot(
    degrees_of_freedom["visitDoF5"],
    ".",
)
xlim = plt.xlim()
plt.plot(xlim, [0, 0], ":")
plt.xlim(xlim)

fig.add_subplot(312)
plt.plot(
    degrees_of_freedom["visitDoF6"],
    ".",
)

fig.add_subplot(313)
plt.plot(
    degrees_of_freedom["visitDoF7"],
    ".",
)

In [ ]:
fig = plt.figure(1)
ax1 = fig.add_subplot(311)
plt.plot(
    camera_hex_corr.visitId-dayobs,
    camera_hex_corr.z,
    ".",
)
plt.xlim(0,700)
plt.ylim(-500, 100)
plt.ylabel("Cam - Z")
plt.grid()
ax1.tick_params(axis='x', labelbottom=False)
plt.title("Hexapod Corrections")

fig = plt.figure(1)
fig.add_subplot(312)
plt.plot(
    m2_hex_corr.visitId-dayobs,
    m2_hex_corr.z,
    ".",
)
plt.ylabel("M2 - Z")
plt.xlabel("Sequence Number")
plt.grid()
fig.subplots_adjust(hspace = 0)
plt.xlim(0,700)
plt.ylim(-500, -100)

In [ ]:
def find_visit_index(visit, data):
    visit_ids = np.array(data.visitId.array)
    return np.where(visit_ids == visit)[0][0]

In [ ]:
def find_visit_indices(visit, data):
    visit_ids = np.array(data.visitId.array)
    return np.where(visit_ids == visit)[0]

In [ ]:
def extract_wfe(data):
    wfe = np.zeros(23)
    for i in range(27):
        value = data[f"nollZernikeValues{i}"]
        index = data[f"nollZernikeIndices{i}"]
        if value is not None:
            wfe[index-4] = value
    return wfe

In [ ]:
find_visit_indices(2026040400141, wavefront_errors)

In [ ]:
index0 = find_visit_index(2026040400141, wavefront_errors)
print(
    wavefront_errors.iloc[index0].visitId, 
    wavefront_errors.iloc[index0].sensorId, 
    wavefront_errors.iloc[index0+1].visitId, 
    wavefront_errors.iloc[index0+1].sensorId,
    wavefront_errors.iloc[index0+2].visitId, 
    wavefront_errors.iloc[index0+2].sensorId, 
    wavefront_errors.iloc[index0+3].visitId, 
    wavefront_errors.iloc[index0+3].sensorId,

)

In [ ]:
wavefront_error = np.array(
    [
        extract_wfe(wavefront_errors.iloc[index0]),
        extract_wfe(wavefront_errors.iloc[index0+1]),
        extract_wfe(wavefront_errors.iloc[index0+2]),
        extract_wfe(wavefront_errors.iloc[index0+3]),
    ]
)

In [ ]:
wavefront_errors.iloc[index0].visitId

In [ ]:
wavefront_error

In [ ]:
ofc_data = OFCData(
    name="lsst",
    log=log,
    config_dir="/net/obs-env/auto_base_packages/ts_config_mttcs/MTAOS/ofc"
)

In [ ]:
sensor_ids=np.array([191, 195, 199, 203])
filter_name='i'
rotation_angle=0.0

In [ ]:
ofc = OFC(
    ofc_data=ofc_data,
    log=log,
)

In [ ]:
ofc.controller.kp = 0.3
ofc.controller.ki = 0
ofc.controller.kd = 0

In [ ]:
new_comp_dof_idx = dict(
    m2HexPos=np.ones(5, dtype=bool),
    camHexPos=np.ones(5, dtype=bool),
    M1M3Bend=np.ones(20, dtype=bool),
    M2Bend=np.ones(20, dtype=bool),
)

new_comp_dof_idx["M1M3Bend"][7:] = False
new_comp_dof_idx["M2Bend"][5:] = False

new_comp_dof_idx = dict(
    m2HexPos=np.ones(5, dtype=bool),
    camHexPos=np.ones(5, dtype=bool),
    M1M3Bend=np.ones(20, dtype=bool),
    M2Bend=np.ones(20, dtype=bool),
)

new_comp_dof_idx["M1M3Bend"][7:] = False
new_comp_dof_idx["M2Bend"][5:] = False

ofc.set_truncation_index(12)
ofc_data.zn_selected = np.array([4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25, 26])
ofc.ofc_data.comp_dof_idx = new_comp_dof_idx
ofc.controller.reset_history()
ofc.state_estimator.refresh_from_ofc_data()

In [ ]:
m2hex_1, camhex_1, m1m3_, m2_1 = ofc.calculate_corrections(
    wfe=wavefront_error,
    sensor_ids=sensor_ids,
    filter_name=filter_name,
    rotation_angle=rotation_angle,
    subtract_intrinsics=False,
    control_vmodes=False,
)

In [ ]:
ofc.lv_dof

In [ ]:
index0 = find_visit_index(2026040400141, degrees_of_freedom)
print(degrees_of_freedom.iloc[index0].visitId)
print(degrees_of_freedom.iloc[index0].visitDoF0,degrees_of_freedom.iloc[index0].visitDoF5)
print(degrees_of_freedom.iloc[index0].visitDoF1,degrees_of_freedom.iloc[index0].visitDoF2)
# These match what is in the PID logs - finally!this_dof = degrees_of_freedom[degrees_of_freedom["visitId"]==2026040400146]

In [ ]:
this_dof['visitDoF0']

In [ ]:
3.939 / 4.112

In [ ]:
3.275 / 3.519

In [ ]:
m2hex_1.correction[2], camhex_1.correction[2]

In [ ]:
degrees_of_freedom[
    degrees_of_freedom.visitId == wavefront_errors.iloc[index0].visitId
].visitDoF0, degrees_of_freedom[
    degrees_of_freedom.visitId == wavefront_errors.iloc[index0].visitId
].visitDoF5, degrees_of_freedom[
    degrees_of_freedom.visitId == wavefront_errors.iloc[index0].visitId
].visitId, degrees_of_freedom[
    degrees_of_freedom.visitId == wavefront_errors.iloc[index0].visitId
].kpGain5, degrees_of_freedom[
    degrees_of_freedom.visitId == wavefront_errors.iloc[index0].visitId
].kiGain5

In [ ]:
degrees_of_freedom[
    degrees_of_freedom.visitId == wavefront_errors.iloc[index0].visitId
].visitDoF0/m2hex_1.correction[2]

In [ ]:
degrees_of_freedom[
    degrees_of_freedom.visitId == wavefront_errors.iloc[index0].visitId
].visitDoF5/camhex_1.correction[2]

In [ ]:
ofc.lv_dof[0], ofc.lv_dof[5]

In [ ]:
valid_visit_ids = np.unique(
    np.array(list(set(degrees_of_freedom.visitId).intersection(set(wavefront_errors.visitId))))
)

In [ ]:
derived_lv_dof = []

for visit_id in valid_visit_ids:
    index0 = find_visit_index(visit_id, wavefront_errors)
    wavefront_error = np.array(
        [
            extract_wfe(wavefront_errors.iloc[index0]),
            extract_wfe(wavefront_errors.iloc[index0+1]),
            extract_wfe(wavefront_errors.iloc[index0+2]),
            extract_wfe(wavefront_errors.iloc[index0+3]),
        ]
    )
    gain_index = find_visit_index(visit_id, degrees_of_freedom)
    if degrees_of_freedom.iloc[gain_index].kpGain0 != 0.3:
        print(f"Skip {visit_id} with gain {degrees_of_freedom.iloc[gain_index].kpGain0}...")
        continue
    print(visit_id)
    m2hex_1, camhex_1, m1m3_, m2_1 = ofc.calculate_corrections(
        wfe=wavefront_error,
        sensor_ids=sensor_ids,
        filter_name=filter_name,
        rotation_angle=rotation_angle,
        subtract_intrinsics=False,
        control_vmodes=False,
    )
    derived_lv_dof.append(ofc.lv_dof)
derived_lv_dof = np.array(derived_lv_dof)
derived_lv_dof_transp = derived_lv_dof.T

In [ ]:
wavefront_error_medium = []

for visit_id in valid_visit_ids:
    index0 = find_visit_index(visit_id, degrees_of_freedom)
    if degrees_of_freedom.iloc[index0].kpGain0 != 0.3:
        print(f"Skip {visit_id} with gain {degrees_of_freedom.iloc[gain_index].kpGain0}...")
        continue
    
    index0 = find_visit_index(visit_id, wavefront_errors)
    wavefront_error = np.array(
        [
            extract_wfe(wavefront_errors.iloc[index0]),
            extract_wfe(wavefront_errors.iloc[index0+1]),
            extract_wfe(wavefront_errors.iloc[index0+2]),
            extract_wfe(wavefront_errors.iloc[index0+3]),
        ]
    )

    wavefront_error_medium.append(np.median(wavefront_error, axis=0))

wavefront_error_medium = np.array(wavefront_error_medium)
wavefront_error_medium_tranp = wavefront_error_medium.T

In [ ]:
degrees_of_freedom.iloc[0].visitDoF5

In [ ]:
index0 = find_visit_index(2026040400146, degrees_of_freedom)
print(degrees_of_freedom.iloc[index0].visitDoF0,degrees_of_freedom.iloc[index0].visitDoF5)
print(degrees_of_freedom.iloc[index0].visitDoF1,degrees_of_freedom.iloc[index0].visitDoF2)
# These match what is in the PID logs - finally!

In [ ]:
published_dof = []

for visit_id in valid_visit_ids:
    index0 = find_visit_index(visit_id, degrees_of_freedom)
    if degrees_of_freedom.iloc[index0].kpGain0 != 0.3:
        print(f"Skip {visit_id} with gain {degrees_of_freedom.iloc[gain_index].kpGain0}...")
        continue
    print(visit_id, degrees_of_freedom.iloc[index0].visitDoF0, degrees_of_freedom.iloc[index0].visitDoF5)
    published_dof.append(
        np.array(
            [
                degrees_of_freedom.iloc[index0].visitDoF0,
                degrees_of_freedom.iloc[index0].visitDoF5,
            ]
        )
    )

published_dof = np.array(published_dof)

In [ ]:
published_dof = published_dof.T

In [ ]:
derived_lv_dof = np.array(derived_lv_dof)

In [ ]:
derived_lv_dof_transp = derived_lv_dof.T

In [ ]:
print(len(published_dof[0]), len(derived_lv_dof_transp[0]))

In [ ]:
for i in range(len(published_dof[0])):
    print(published_dof[0][i], derived_lv_dof_transp[0][i], published_dof[0][i]/derived_lv_dof_transp[0][i])

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

axes[0].plot(
    published_dof[0],
    derived_lv_dof_transp[0],
    "."
)
axes[0].plot(
    [-200, 200],
    [-200, 200],
    "--",
)
axes[0].set_xlim(-200, 200)
axes[0].set_ylim(-200, 200)
axes[0].set_xlabel("Published DoF 0 (M2 dz)")
axes[0].set_ylabel("Calculated DoF 0 (M2 dz)")

axes[1].plot(
    published_dof[1],
    derived_lv_dof_transp[5],
    "."    
)
axes[1].plot(
    [-200, 200],
    [-200, 200],
    "--",
)
axes[1].set_xlim(-200, 200)
axes[1].set_ylim(-200, 200)
axes[1].set_xlabel("Published DoF 5 (Cam dz)")
axes[1].set_ylabel("Calculated DoF 5 (Cam dz)")


In [ ]:
print(published_dof[0])
print(derived_lv_dof_transp[0])


In [ ]:
plt.plot(
    published_dof[1],
    derived_lv_dof_transp[5],
    "."    
)
plt.plot(
    [-200, 200],
    [-200, 200],
    "--",
)
plt.xlim(-200, 200)
plt.ylim(-200, 200)

In [ ]:
fig = plt.figure(1)
#fig.add_subplot(121)

plt.plot(
    wavefront_error_medium_tranp[0],
    derived_lv_dof_transp[0],
    "."
)

# fig.add_subplot(122)
plt.plot(
    wavefront_error_medium_tranp[0],
    derived_lv_dof_transp[5],
    "."
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# Left: zk4 vs dof0
axes[0].scatter(
    published_dof[0],
    wavefront_error_medium_tranp[0],
    s=9,
    alpha=0.5,
)
axes[0].scatter(
    derived_lv_dof_transp[0],
    wavefront_error_medium_tranp[0],
    s=9,
    color='blue',
)
xline0 = np.linspace(published_dof[0].min(), published_dof[0].max(), 100)
# slope = -0.17
# slope = 0.0
axes[0].plot(np.zeros_like(xline0), slope * xline5-0.2, "--", label=f"DoF off")


axes[0].set_xlabel(r"Instantaneous $\Delta$M2 dz [um]")
axes[0].set_ylabel("dz (1,4) [um]")
axes[0].set_xlim(-75, 75)
axes[0].set_ylim(-1.5, 1)
axes[0].legend()

# Right: zk4 vs dof5
axes[1].scatter(
   published_dof[1],
   wavefront_error_medium_tranp[0],
   s=9,
    alpha=0.5,
)
axes[1].scatter(
    derived_lv_dof_transp[5],
    wavefront_error_medium_tranp[0],
    s=9,
    color='blue',
)

xline5 = np.linspace(published_dof[1].min(), published_dof[1].max(), 100)
# slope = -0.075
slope =-0.05
axes[1].plot(xline5, slope * xline5-0.2, "--", label=f"{slope=:0.3f}")

axes[1].set_xlim(-75, 75)
axes[1].set_ylim(-1.5, 1)
axes[1].set_xlabel(r"Instantaneous $\Delta$Cam dz [um]")
axes[1].legend()

plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# Left: zk4 vs dof0
axes[0].scatter(
    published_dof[0],
    wavefront_error_medium_tranp[0],
    s=9,
    alpha=0.5,
)
axes[0].scatter(
    derived_lv_dof_transp[0],
    wavefront_error_medium_tranp[0],
    s=9,
    color='blue',
)
xline0 = np.linspace(published_dof[0].min(), published_dof[0].max(), 100)
slope = -0.17
# slope = 0.0
axes[0].plot(xline5, slope * xline5-0.2, "--", label=f"{slope=:0.3f}")


axes[0].set_xlabel(r"Instantaneous $\Delta$M2 dz [um]")
axes[0].set_ylabel("dz (1,4) [um]")
axes[0].set_xlim(-75, 75)
axes[0].set_ylim(-1.5, 1)
axes[0].legend()

# Right: zk4 vs dof5
axes[1].scatter(
   published_dof[1],
   wavefront_error_medium_tranp[0],
   s=9,
    alpha=0.5,
)
axes[1].scatter(
    derived_lv_dof_transp[5],
    wavefront_error_medium_tranp[0],
    s=9,
    color='blue',
)

xline5 = np.linspace(published_dof[1].min(), published_dof[1].max(), 100)
slope = -0.075
# slope =-0.05
axes[1].plot(xline5, slope * xline5-0.2, "--", label=f"{slope=:0.3f}")

axes[1].set_xlim(-75, 75)
axes[1].set_ylim(-1.5, 1)
axes[1].set_xlabel(r"Instantaneous $\Delta$Cam dz [um]")
axes[1].legend()

plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# Left: zk4 vs dof0
axes[0].scatter(
    published_dof[0],
    wavefront_error_medium_tranp[0],
    s=9,
    alpha=0.5,
)
axes[0].scatter(
    derived_lv_dof_transp[0],
    wavefront_error_medium_tranp[0],
    s=9,
    color='blue',
)
xline0 = np.linspace(published_dof[0].min(), published_dof[0].max(), 100)
slope = -0.17
# slope = 0.0
axes[0].plot(xline5, slope * xline5-0.2, "--", label=f"{slope=:0.3f}")


axes[0].set_xlabel(r"Instantaneous $\Delta$M2 dz [um]")
axes[0].set_ylabel("dz (1,4) [um]")
axes[0].set_xlim(-75, 75)
axes[0].set_ylim(-1.5, 1)
axes[0].legend()

# Right: zk4 vs dof5
axes[1].scatter(
   published_dof[1],
   wavefront_error_medium_tranp[0],
   s=9,
    alpha=0.5,
)
axes[1].scatter(
    derived_lv_dof_transp[5],
    wavefront_error_medium_tranp[0],
    s=9,
    color='blue',
)

xline5 = np.linspace(published_dof[1].min(), published_dof[1].max(), 100)
slope = -0.075
# slope =-0.05
axes[1].plot(xline5, slope * xline5-0.2, "--", label=f"{slope=:0.3f}")

axes[1].set_xlim(-75, 75)
axes[1].set_ylim(-1.5, 1)
axes[1].set_xlabel(r"Instantaneous $\Delta$Cam dz [um]")
axes[1].legend()

plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# Left: zk4 vs dof0
axes[0].scatter(
    published_dof[0],
    wavefront_error_medium_tranp[0],
    s=9
)
axes[0].scatter(
    derived_lv_dof_transp[0],
    wavefront_error_medium_tranp[0],
    s=9,
    color='blue',
)
xline0 = np.linspace(published_dof[0].min(), published_dof[0].max(), 100)
axes[0].plot(xline0, -0.015 * xline0, "--", label="slope = 0.015")
axes[0].set_xlabel(r"Instantaneous $\Delta$M2 dz [um]")
axes[0].set_ylabel("dz (1,4) [um]")
axes[0].set_xlim(-75, 75)
axes[0].set_ylim(-1.5, 1)
axes[0].legend()

# Right: zk4 vs dof5
#axes[1].scatter(
#    published_dof[1],
#    wavefront_error_medium_tranp[0],
#    s=9
#)
axes[1].scatter(
    derived_lv_dof_transp[5],
    wavefront_error_medium_tranp[0],
    s=9,
    color='blue',
)

xline5 = np.linspace(published_dof[1].min(), published_dof[1].max(), 100)
axes[1].plot(xline5, -0.05 * xline5-0.2, "--", label="slope = 0.015")
axes[1].set_xlim(-75, 75)
axes[1].set_ylim(-1.5, 1)
axes[1].set_xlabel(r"Instantaneous $\Delta$Cam dz [um]")
axes[1].legend()

plt.title("All Cam DoFs")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# Left: zk4 vs dof0
axes[0].scatter(
    published_dof[0],
    wavefront_error_medium_tranp[0],
    s=9
)
axes[0].scatter(
    derived_lv_dof_transp[0],
    wavefront_error_medium_tranp[0],
    s=9,
    color='blue',
)
xline0 = np.linspace(published_dof[0].min(), published_dof[0].max(), 100)
axes[0].plot(xline0, -0.015 * xline0, "--", label="slope = 0.015")
axes[0].set_xlabel(r"Instantaneous $\Delta$M2 dz [um]")
axes[0].set_ylabel("dz (1,4) [um]")
axes[0].set_xlim(-75, 75)
axes[0].set_ylim(-1.5, 1)
axes[0].legend()

# Right: zk4 vs dof5
#axes[1].scatter(
#    published_dof[1],
#    wavefront_error_medium_tranp[0],
#    s=9
#)
axes[1].scatter(
    derived_lv_dof_transp[5],
    wavefront_error_medium_tranp[0],
    s=9,
    color='blue',
)

xline5 = np.linspace(published_dof[1].min(), published_dof[1].max(), 100)
axes[1].plot(xline5, -0.05 * xline5-0.2, "--", label="slope = 0.015")
axes[1].set_xlim(-60, 60)
axes[1].set_ylim(-1.5, 1)
axes[1].set_xlabel(r"Instantaneous $\Delta$Cam dz [um]")
axes[1].legend()

plt.title("All Cam DoFs")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(12, 5), sharey=True)
plt.imshow(derived_lv_dof_transp)

In [ ]:
np.save(
    "wavefront_error_medium_tranp.npy",
    wavefront_error_medium_tranp,
)

In [ ]:
np.save(
    "derived_lv_dof_transp.npy",
    derived_lv_dof_transp,
)

In [ ]:
np.save(
    "visit_ids.npy",
    np.unique(wavefront_errors.visitId),
)

In [ ]:
np.save(
    "derived_lv_dof_transp_cam_dz.npy",
    derived_lv_dof_transp,
)

In [ ]:
np.save(
    "derived_lv_dof_transp_all_cam_dofs.npy",
    derived_lv_dof_transp,
)